In [ ]:
from collections import defaultdict, deque

#parse context free grammar from string format to dict format
def parseCFG(cfg):
    #store nonterminals
    grammar = defaultdict(list)
    #splits input
    lines = cfg.strip().split("\n")
    
    #split right and left
    for line in lines:
        left, right = line.split("->")
        #remove whitespace
        left = left.strip()
        #separate productions by |
        productions = right.strip().split("|")
        for prod in productions:
            #remove whitespace
            prod = prod.strip()
            #empty
            if prod == "empty":
                grammar[left].append([])
            else:
                #store as a list
                grammar[left].append(list(prod))
    return grammar

# CFG to NPDA 
def cfgToNPDA(grammar):
    #NPDA states
    states = ["q0", "q1", "q2"]
    #accept state
    final = {"q2"}

    transitions = []

    # start is S
    transitions.append("q0-empty-z-Sz->q1")

    # CFG production
    for A in grammar:
        for production in grammar[A]:
            if production == []:
                rhs = "empty"
            else:
                rhs = "".join(production)
            transitions.append(f"q1-empty-{A}-{rhs}->q1")

    # match terminals
    for terminal in ["a", "b"]:
        transitions.append(f"q1-{terminal}-{terminal}-empty->q1")

    # accept
    transitions.append("q1-empty-z-z->q2")

    # create header with states and final states marked with 'f' so we can parse them back later
    state_tokens = [s + ("f" if s in final else "") for s in states]
    return ",".join(state_tokens) + "," + ",".join(transitions)

# parse NPDA string back into states and transition 
def parseNPDA(npda_str):
    parts = npda_str.split(",")

    #store states and accept states
    states = []
    final_states = set()
    i = 0
    while i < len(parts):
        token = parts[i]
        if "-" in token:
            break
        if token.endswith("f"):
            name = token[:-1]
            final_states.add(name)
        else:
            name = token
        states.append(name)
        i += 1

    #used lambda to create a nested dict of lists for transitions
    transitions = defaultdict(lambda: defaultdict(list))
    for t in parts[i:]:
        #split on '->'
        # Split transition into left-hand side and destination state
        left_side, next_state = t.split("->")
        current_state, input_symbol, stack_top, push_string = left_side.split("-")
        input_symbol = None if input_symbol == "empty" else input_symbol
        push_symbols = [] if push_string == "empty" else list(push_string)
        transitions[current_state][(input_symbol, stack_top)].append((next_state, push_symbols))
    return states, final_states, transitions

# BFS with visited set avoids infinite loops 
def simulateNPDA(npda_str, string):
    #parse NPDA string into states and transitions
    _, final_states, transitions = parseNPDA(npda_str)

    inputLength = len(string)

    # start at q0 with z on the stack
    initial = ("q0", 0, ("z",))
    queue = deque([initial])
    #track visited states to avoid infinite loops
    visited = {initial}

    while queue:
        state, index, stack = queue.popleft()

        #accept if in final state and input is done
        if state in final_states and index == inputLength:
            return True

        #nothing left to pop
        if not stack:
            continue

        #reject if stack is too big
        remaining = inputLength - index
        if len(stack) > remaining * 2 + 10:
            continue

        #get top of stack and pop
        stack_top = stack[-1]
        stack_rest = stack[:-1]

        # try all transitions for current state and stack top
        for (inp_sym, top), dests in transitions[state].items():
            if top != stack_top:
                continue

            #check if match terminal
            if inp_sym is not None:
                if index >= inputLength or string[index] != inp_sym:
                    continue
                new_index = index + 1
            else:
                #epsilon
                new_index = index

            for (dst, push_syms) in dests:
                #push symbols reversed 
                new_stack = stack_rest + tuple(push_syms[::-1])
                config = (dst, new_index, new_stack)
                #avoid infinite loops
                if config not in visited:
                    visited.add(config)
                    queue.append(config)

    #nothing worked, reject
    return False

#used ai for testing and formatting
#run strings through npda and print results
def run(grammar, strings):
    # build the NPDA string from the grammar
    npda_str = cfgToNPDA(grammar)
    print("NPDA:")
    print(npda_str)
    for s in strings:
        # use the machine on each string
        result = simulateNPDA(npda_str, s)
        #print result
        print(f"{s} -> {'accept' if result else 'reject'}")

#Main: test cases 1-5
if __name__ == "__main__":
    print("\nTest Case 1:")
    cfg1 = """
    S -> AA
    A -> aAb | empty
    """
    run(parseCFG(cfg1), ["aababb", "aaaaabbbbb"])

    print("\nTest Case 2:")
    cfg2 = """
    S -> U|V
    U -> XAX|UU
    V -> XBX|VV
    X -> aXb|bXa|XX|empty
    A -> aA|a
    B -> bB|b
    """
    run(parseCFG(cfg2), ["baabba", "aaabbb"])

    print("\nTest Case 3:")
    cfg3 = """
    S -> UaU
    U -> bUaUa|aUbUa|aUaUb|UU|empty
    """
    run(parseCFG(cfg3), ["bbaabaaaaa", "abaaba"])

    print("\nTest Case 4:")
    cfg4 = """
    S -> XaXaX
    X -> aXb|bXa|XX|empty
    """
    run(parseCFG(cfg4), ["baabaa", "abaabab"])

    print("\nTest Case 5:")
    cfg5 = """
    S -> UV
    U -> XAX|UU
    X -> aXb|bXa|XX|empty
    A -> aA|a
    V -> aV|bV|empty
    """
    run(parseCFG(cfg5), ["bbbaab", "aababbb"])


Test Case 1:
NPDA:
q0,q1,q2f,q0-empty-z-Sz->q1,q1-empty-S-AA->q1,q1-empty-A-aAb->q1,q1-empty-A-empty->q1,q1-a-a-empty->q1,q1-b-b-empty->q1,q1-empty-z-z->q2
aababb -> reject
aaaaabbbbb -> accept

Test Case 2:
NPDA:
q0,q1,q2f,q0-empty-z-Sz->q1,q1-empty-S-U->q1,q1-empty-S-V->q1,q1-empty-U-XAX->q1,q1-empty-U-UU->q1,q1-empty-V-XBX->q1,q1-empty-V-VV->q1,q1-empty-X-aXb->q1,q1-empty-X-bXa->q1,q1-empty-X-XX->q1,q1-empty-X-empty->q1,q1-empty-A-aA->q1,q1-empty-A-a->q1,q1-empty-B-bB->q1,q1-empty-B-b->q1,q1-a-a-empty->q1,q1-b-b-empty->q1,q1-empty-z-z->q2
baabba -> reject
aaabbb -> reject

Test Case 3:
NPDA:
q0,q1,q2f,q0-empty-z-Sz->q1,q1-empty-S-UaU->q1,q1-empty-U-bUaUa->q1,q1-empty-U-aUbUa->q1,q1-empty-U-aUaUb->q1,q1-empty-U-UU->q1,q1-empty-U-empty->q1,q1-a-a-empty->q1,q1-b-b-empty->q1,q1-empty-z-z->q2
bbaabaaaaa -> accept
abaaba -> reject

Test Case 4:
NPDA:
q0,q1,q2f,q0-empty-z-Sz->q1,q1-empty-S-XaXaX->q1,q1-empty-X-aXb->q1,q1-empty-X-bXa->q1,q1-empty-X-XX->q1,q1-empty-X-empty->q1,q1-a-a-empty-